<div style="background:linear-gradient(120deg,#EDE7FF 0%,#E3F0FF 55%,#E6FBF7 100%); color:#2A1B5C; padding:22px 26px; border-radius:14px; border:1px solid #DCD3FF; margin-bottom:8px;">

<h1 style="margin:0 0 6px 0; color:#3B1F7A">03 · HCP 360: one row per doctor</h1>

<p style="margin:0 0 10px 0; font-size:1.1em; color:#4C3A8C">Unifying prescriptions, registry details, payments and state trends into a single doctor profile</p>

| | |
|---|---|
| **Goal** | Build the master table every later step uses: targeting scores, dashboards, pre-visit briefs, the AI agent |
| **Code** | `src/glp1_intel/data_processing/hcp360.py` |
| **Inputs** | Part D 2024 (all drugs + GLP-1 subset) · NPPES registry · Open Payments · Medicaid SDUD |
| **Output** | `data/03_processed/hcp_360.parquet` (316,559 doctors × 61 columns) |
| **Skills** | Data unification · keys and joins · feature engineering · data validation |

</div>

<div style="margin:6px 0 4px 0"><b>How to read this notebook:</b><br><span style="background:#F3EEFF; color:#3B1F7A; border-left:4px solid #8B5CF6; padding:3px 10px; border-radius:6px; margin-right:6px; display:inline-block; margin-bottom:4px">🟣 Section</span><span style="background:#FFF4E8; color:#7C2D12; border-left:4px solid #FB923C; padding:3px 10px; border-radius:6px; margin-right:6px; display:inline-block; margin-bottom:4px">💡 Concept</span><span style="background:#E9FBF8; color:#134E4A; border-left:4px solid #14B8A6; padding:3px 10px; border-radius:6px; margin-right:6px; display:inline-block; margin-bottom:4px">🛠️ What the next code does</span><span style="background:#EAF4FF; color:#1E3A8A; border-left:4px solid #3B82F6; padding:3px 10px; border-radius:6px; margin-right:6px; display:inline-block; margin-bottom:4px">📊 Reading the output</span><span style="background:#FFF0F3; color:#881337; border-left:4px solid #F43F5E; padding:3px 10px; border-radius:6px; margin-right:6px; display:inline-block; margin-bottom:4px">⚠️ Watch out</span><span style="background:#EAFBF1; color:#065F46; border-left:4px solid #10B981; padding:3px 10px; border-radius:6px; margin-right:6px; display:inline-block; margin-bottom:4px">🎯 Key takeaways</span></div>

<div style="border-left:6px solid #FB923C; background:#FFF4E8; color:#7C2D12; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">💡 Concept · What is an HCP 360, and why one row per doctor?</b>

**HCP** = healthcare professional. An **HCP 360** (or "single customer view") gathers everything known about one doctor into **one row**,
the same idea banks use to see each customer across accounts, cards and loans.

Each source describes the doctor from a different angle:

| Angle | Source | Example columns |
|---|---|---|
| **What they prescribe** | Medicare Part D 2024 | GLP-1 claims by brand, other diabetes drugs, practice size |
| **Who they are** | NPPES registry | credential, sex, practice ZIP, taxonomy, deactivated? |
| **How engaged industry is** | Open Payments | Lilly / Novo payments, meals, speaker fees |
| **Where they work** | Medicaid SDUD (state level) | obesity-GLP-1 growth in their state |

The glue is the **NPI** (National Provider Identifier): one unique 10-digit ID per doctor, used by every US health dataset.

</div>

<div style="border-left:6px solid #8B5CF6; background:#F3EEFF; color:#3B1F7A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

## 1 · Who belongs in the table? Defining the universe

</div>

<div style="border-left:6px solid #FB923C; background:#FFF4E8; color:#7C2D12; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">💡 Concept</b>

A table of only GLP-1 prescribers would miss the most interesting doctors: those who **treat many diabetes patients but haven't started GLP-1s**.
In sales optimisation these **non-adopters** (or low adopters) are often the biggest growth opportunity.

So the universe is: **every 2024 Part D prescriber who wrote a GLP-1 *or* any other diabetes drug**
(metformin, SGLT2 inhibitors, DPP-4 inhibitors, insulins, sulfonylureas, pioglitazone…).
Insulin syringes and pumps are excluded: they're supplies, not drugs.

</div>

<div style="border-left:6px solid #F43F5E; background:#FFF0F3; color:#881337; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">⚠️ Watch out</b>

**SGLT2 inhibitors** (empagliflozin, dapagliflozin) are also prescribed for **heart failure and kidney disease**.
A cardiologist with many SGLT2 claims isn't necessarily treating diabetes, so we keep classes separate (`dm_sglt2_claims`, `dm_metformin_claims`, …) instead of only one total.

</div>

<div style="border-left:6px solid #14B8A6; background:#E9FBF8; color:#134E4A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">🛠️ What the next code does</b>

`hcp360.build()` runs the whole pipeline and caches every step:
1. **One pass over the 4.4 GB Part D file** (DuckDB, ~25 s): practice size + claims per diabetes class for all 1.1M prescribers.
2. Keep the diabetes / GLP-1 universe, then **stream NPPES** for exactly those NPIs (~4 min, first run only).
3. **LEFT JOIN** everything on NPI, then add derived features (shares, deciles, adopter flag).

</div>

In [1]:
# ── Imports ─────────────────────────────────────────────────────────────────
import duckdb
import pandas as pd

from glp1_intel.notebook_style import show
from glp1_intel.data_processing import hcp360

# ── Build (or load the cached) HCP 360 table ────────────────────────────────
hcp = hcp360.build()          # first run ~8 min; afterwards only the final join re-runs
H   = f"read_parquet('{hcp360.HCP_360.as_posix()}')"     # table expression for DuckDB queries

print(f"{len(hcp):,} doctors × {hcp.shape[1]} columns")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

316,559 doctors × 61 columns


In [2]:
universe = duckdb.sql(f"""
    SELECT  CASE WHEN glp1_adopter THEN 'GLP-1 prescriber' ELSE 'Diabetes only (no GLP-1)' END AS doctor_type,
            COUNT(*)                         AS doctors,
            median(diabetes_total_claims)    AS median_diabetes_claims,
            SUM(dm_non_glp1_claims)          AS other_diabetes_claims,
            SUM(glp1_claims)                 AS glp1_claims
    FROM    {H}
    GROUP BY ALL
""").df()
show(universe, "The HCP 360 universe")

doctor_type,doctors,median_diabetes_claims,other_diabetes_claims,glp1_claims
Diabetes only (no GLP-1),"116,728",37,"9,753,922",0
GLP-1 prescriber,"199,831",296,"74,866,496","19,620,006"


<div style="border-left:6px solid #3B82F6; background:#EAF4FF; color:#1E3A8A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">📊 Reading the output</b>

- **199,831 GLP-1 prescribers** (exactly the notebook-02 count ✅) plus **116,728 doctors who treat diabetes but wrote no GLP-1** in 2024.
- Non-adopters are mostly **smaller** diabetes prescribers (median 37 diabetes claims vs 296), but not all, as section 5 shows.

</div>

<div style="border-left:6px solid #8B5CF6; background:#F3EEFF; color:#3B1F7A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

## 2 · Joins: why LEFT JOIN, and how well did each source match?

</div>

<div style="border-left:6px solid #FB923C; background:#FFF4E8; color:#7C2D12; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">💡 Concept · LEFT vs INNER JOIN, and blank ≠ zero</b>

| Join | Keeps | Risk |
|---|---|---|
| `INNER JOIN` | only doctors found in **both** tables | silently **drops** doctors missing from one source |
| `LEFT JOIN` | **every** doctor in the left table; blanks where the right table has no match | none, as long as we treat blanks correctly |

We always LEFT JOIN onto the universe, so no doctor disappears because, say, they received no industry payments.

**A blank is not a zero.** A missing NPPES record means *unknown*, but missing Open Payments means *no payments reported*.
We record which is which, and only fill zeros where zero is the true meaning (e.g. claims of a brand never prescribed).

</div>

In [3]:
pay_cols = [c for c in hcp.columns if c.startswith("op_")]

coverage = pd.DataFrame([
    {"source": "Part D prescribing (universe)", "matched_pct": 100.0},
    {"source": "NPPES registry",                "matched_pct": 100 * hcp["practice_zip5"].notna().mean()},
    {"source": "State Medicaid context",        "matched_pct": 100 * hcp["state_medicaid_obesity_rx_2025"].notna().mean()},
    {"source": "Open Payments (any Lilly/Novo payment)",
     "matched_pct": 100 * (hcp[["op_lilly_payments", "op_novo_payments"]].sum(axis=1) > 0).mean() if pay_cols else float("nan")},
])
show(coverage, "Join coverage: % of doctors with a match in each source")

source,matched_pct
Part D prescribing (universe),100.0
NPPES registry,99.6
State Medicaid context,99.9
Open Payments (any Lilly/Novo payment),38.7


<div style="border-left:6px solid #3B82F6; background:#EAF4FF; color:#1E3A8A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">📊 Reading the output</b>

- **NPPES: 99.6% matched.** The few gaps are NPIs that were removed or changed since 2024.
- **State context: ~100%**; only territories without Medicaid SDUD rows are missing.
- **Open Payments: ~39%** of doctors received at least one Lilly or Novo payment in 2024–2025 (Lilly ~28%, Novo ~33%).
  A doctor with no row there simply received **no reported payment**, which is a real 0, not unknown, so these columns are filled with 0.

</div>

<div style="border-left:6px solid #8B5CF6; background:#F3EEFF; color:#3B1F7A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

## 3 · Validate before you trust

</div>

<div style="border-left:6px solid #14B8A6; background:#E9FBF8; color:#134E4A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">🛠️ What the next code does</b>

Three automatic checks, each one an `assert` that stops the notebook if it fails:
1. **Primary key:** every NPI appears exactly once.
2. **No double counting:** total GLP-1 claims equal the notebook-02 total (**19,620,006**). A bad join would inflate this.
3. **Adopter count** equals the notebook-02 prescriber count (**199,831**).

</div>

In [4]:
assert hcp["npi"].is_unique,                               "Duplicate NPIs: a join multiplied rows"
assert hcp["glp1_claims"].sum() == 19_620_006,             "GLP-1 claims changed: check the joins"
assert hcp["glp1_adopter"].sum() == 199_831,               "Adopter count differs from notebook 02"

print("✅ 1 row per NPI   ✅ GLP-1 claims reconcile   ✅ adopter count reconciles")

✅ 1 row per NPI   ✅ GLP-1 claims reconcile   ✅ adopter count reconciles


<div style="border-left:6px solid #FB923C; background:#FFF4E8; color:#7C2D12; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">💡 Concept · Why reconcile totals?</b>

Reconciling totals after every join is the cheapest bug detector in data engineering.
A many-to-many join (e.g. a doctor with two NPPES rows) silently **multiplies** claims, and the dashboard would look fine but be wrong.

</div>

<div style="border-left:6px solid #8B5CF6; background:#F3EEFF; color:#3B1F7A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

## 4 · What is in a row? The column groups

</div>

In [5]:
groups = {
    "Identity":            ["npi", "last_name", "first_name", "city", "state", "specialty"],
    "Registry (NPPES)":    ["credential", "sex", "practice_zip5", "taxonomy", "npi_year", "npi_deactivated"],
    "Practice size":       ["all_drug_claims", "all_drug_cost", "n_drugs"],
    "Other diabetes drugs": [c for c in hcp.columns if c.startswith("dm_")],
    "GLP-1 prescribing":   ["glp1_claims", "glp1_cost", "tirzepatide_claims", "semaglutide_claims", "dulaglutide_claims",
                            "mounjaro_claims", "zepbound_claims", "ozempic_claims", "wegovy_claims", "trulicity_claims"],
    "Industry payments":   pay_cols or ["(added when Open Payments finishes)"],
    "State context":       ["state_medicaid_obesity_rx_2025", "state_medicaid_obesity_growth_24_25"],
    "Derived features":    ["diabetes_total_claims", "glp1_adopter", "glp1_share_of_diabetes", "tirzepatide_share",
                            "lilly_glp1_share", "glp1_decile", "diabetes_decile"],
}
show(pd.DataFrame([{"group": g, "n_columns": len(c), "columns": ", ".join(c)} for g, c in groups.items()]),
     "HCP 360 column groups (full definitions: docs/hcp_360_data_dictionary.md)")

group,n_columns,columns
Identity,6,"npi, last_name, first_name, city, state, specialty"
Registry (NPPES),6,"credential, sex, practice_zip5, taxonomy, npi_year, npi_deactivated"
Practice size,3,"all_drug_claims, all_drug_cost, n_drugs"
Other diabetes drugs,8,"dm_non_glp1_claims, dm_sglt2_claims, dm_dpp4_claims, dm_insulin_claims, dm_su_claims, dm_tzd_claims, dm_metformin_claims, dm_other_oral_claims"
GLP-1 prescribing,10,"glp1_claims, glp1_cost, tirzepatide_claims, semaglutide_claims, dulaglutide_claims, mounjaro_claims, zepbound_claims, ozempic_claims, wegovy_claims, trulicity_claims"
Industry payments,11,"op_lilly_payments, op_lilly_usd, op_lilly_glp1_payments, op_lilly_meals, op_lilly_speaker_consult_usd, op_novo_payments, op_novo_usd, op_novo_glp1_payments, op_novo_meals, op_novo_speaker_consult_usd, op_years"
State context,2,"state_medicaid_obesity_rx_2025, state_medicaid_obesity_growth_24_25"
Derived features,7,"diabetes_total_claims, glp1_adopter, glp1_share_of_diabetes, tirzepatide_share, lilly_glp1_share, glp1_decile, diabetes_decile"


<div style="border-left:6px solid #FB923C; background:#FFF4E8; color:#7C2D12; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">💡 Concept · Feature engineering: ratios beat raw counts</b>

Raw counts don't compare well across doctors of different size, so we derive **ratios**:

| Feature | Formula | Question it answers |
|---|---|---|
| `glp1_share_of_diabetes` | GLP-1 claims ÷ all diabetes claims | *How far has this doctor adopted GLP-1s?* |
| `tirzepatide_share` | tirzepatide ÷ (tirzepatide + semaglutide) | *Lilly's molecule or Novo's?* |
| `lilly_glp1_share` | (tirzepatide + dulaglutide) ÷ all GLP-1 | *Company-level share of this doctor* |
| `glp1_decile` | rank GLP-1 volume into 10 groups (1 = top 10%) | *How big a GLP-1 prescriber?* |
| `diabetes_decile` | same, for all diabetes claims | *How big is their diabetes patient base: the **potential**?* |

**Potential vs current value** is the core idea of pharma targeting: `diabetes_decile` ≈ potential, `glp1_share_of_diabetes` ≈ how much of it is already GLP-1.

</div>

<div style="border-left:6px solid #8B5CF6; background:#F3EEFF; color:#3B1F7A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

## 5 · First insights from the unified table

</div>

In [6]:
by_spec = duckdb.sql(f"""
    SELECT  specialty,
            COUNT(*)                                                          AS doctors,
            ROUND(100 * AVG(glp1_adopter::INT), 1)                            AS pct_prescribing_glp1,
            ROUND(100 * SUM(glp1_claims) / SUM(diabetes_total_claims), 1)     AS glp1_pct_of_diabetes_rx
    FROM    {H}
    GROUP BY ALL
    HAVING  COUNT(*) > 3000
    ORDER BY SUM(diabetes_total_claims) DESC
    LIMIT 10
""").df()
show(by_spec, "5a · GLP-1 adoption by specialty (top 10 by diabetes volume)")

specialty,doctors,pct_prescribing_glp1,glp1_pct_of_diabetes_rx
Family Practice,"82,338",77.8,18.1
Internal Medicine,"62,997",71.4,16.9
Nurse Practitioner,"86,058",60.2,21.3
Endocrinology,"6,076",96.8,26.7
Physician Assistant,"26,851",65.2,22.4
General Practice,"4,723",50.5,8.0
Cardiology,"13,524",17.1,10.0
Nephrology,"6,784",16.9,7.9
Interventional Cardiology,"3,051",13.0,8.5
Student in an Organized Health Care Education/Training Program,"5,971",33.3,20.1


<div style="border-left:6px solid #3B82F6; background:#EAF4FF; color:#1E3A8A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">📊 Reading the output</b>

- **Endocrinologists:** 97% prescribe GLP-1s, and GLP-1s are 27% of their diabetes prescriptions. They're the most "converted" group.
- **Primary care** (family practice, internal medicine, NPs, PAs): 60–78% adopt, and GLP-1s are only ~17–22% of diabetes Rx. Lots of room left.
- **Cardiology and nephrology: only ~17% prescribe GLP-1s**, and GLP-1s are 8–10% of their diabetes-drug claims (mostly SGLT2s for heart and kidney disease).
  Since several GLP-1s now have cardiovascular and kidney evidence, these specialists are a **new cardio-metabolic audience** to watch.

</div>

In [7]:
loyalty = duckdb.sql(f"""
    SELECT  CASE WHEN tirzepatide_share IS NULL THEN '4 · no tirzepatide or semaglutide'
                 WHEN tirzepatide_share >= 0.8  THEN '1 · tirzepatide-loyal (≥ 80%)'
                 WHEN tirzepatide_share <= 0.2  THEN '3 · semaglutide-loyal (≤ 20%)'
                 ELSE                                '2 · mixed (20–80%)' END                AS segment,
            COUNT(*)                                                                         AS doctors,
            ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)                                 AS pct_doctors,
            SUM(glp1_claims)                                                                 AS glp1_claims,
            ROUND(100 * SUM(glp1_claims) / SUM(SUM(glp1_claims)) OVER (), 1)                 AS pct_claims
    FROM    {H}
    WHERE   glp1_adopter
    GROUP BY ALL
    ORDER BY segment
""").df()
show(loyalty, "5b · Brand loyalty: tirzepatide vs semaglutide")

segment,doctors,pct_doctors,glp1_claims,pct_claims
1 · tirzepatide-loyal (≥ 80%),"11,705",5.9,"510,784",2.6
2 · mixed (20–80%),"72,113",36.1,"11,734,647",59.8
3 · semaglutide-loyal (≤ 20%),"101,748",50.9,"7,036,527",35.9
4 · no tirzepatide or semaglutide,"14,265",7.1,"338,048",1.7


<div style="border-left:6px solid #3B82F6; background:#EAF4FF; color:#1E3A8A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">📊 Reading the output</b>

- **Half of GLP-1 prescribers (51%) are semaglutide-loyal**, while only 6% are tirzepatide-loyal.
- But **mixed prescribers** (36% of doctors) write **60% of all GLP-1 claims**: big prescribers try both molecules.
- Commercial reading: **mixed** doctors are the swing audience (already familiar with tirzepatide), and **semaglutide-loyal** doctors are the conversion challenge.

</div>

In [8]:
opportunity = duckdb.sql(f"""
    SELECT  CASE WHEN diabetes_decile <= 2 THEN '1 · High potential (top 20% diabetes volume)'
                 WHEN diabetes_decile <= 5 THEN '2 · Medium potential'
                 ELSE                           '3 · Low potential' END                      AS potential,
            COUNT(*) FILTER (WHERE glp1_share_of_diabetes < 0.10 OR NOT glp1_adopter)   AS low_glp1_use_doctors,
            COUNT(*) FILTER (WHERE glp1_share_of_diabetes >= 0.10)                      AS established_glp1_doctors,
            SUM(dm_non_glp1_claims) FILTER (WHERE glp1_share_of_diabetes < 0.10 OR NOT glp1_adopter)
                                                                                         AS other_dm_claims_low_users
    FROM    {H}
    GROUP BY ALL
    ORDER BY potential
""").df()
show(opportunity, "5c · Opportunity matrix: potential vs current GLP-1 use (< 10% of diabetes Rx = low)")

potential,low_glp1_use_doctors,established_glp1_doctors,other_dm_claims_low_users
1 · High potential (top 20% diabetes volume),"14,301","49,011","14,659,122"
2 · Medium potential,"27,489","67,478","7,460,389"
3 · Low potential,"101,956","56,324","4,463,744"


<div style="border-left:6px solid #3B82F6; background:#EAF4FF; color:#1E3A8A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">📊 Reading the output</b>

- **~14,000 high-potential doctors** (top 20% diabetes volume) use GLP-1s for **under 10%** of their diabetes prescriptions, and together they write **~14.7 million** other diabetes prescriptions.
- That's the classic **"high potential, low share"** quadrant that sales-optimisation teams prioritise. Notebook 05 turns this into a proper targeting score with reasons.

</div>

<div style="border-left:6px solid #F43F5E; background:#FFF0F3; color:#881337; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">⚠️ Watch out</b>

Not every low-GLP-1 doctor *should* prescribe more. Patients may have contraindications, cost barriers or other priorities.
The table shows **where to start a conversation**, not what a doctor should prescribe. Targeting is about **relevance of information**, never pressure.

</div>

In [9]:
by_decile = duckdb.sql(f"""
    SELECT  glp1_decile,
            COUNT(*)                                       AS doctors,
            ROUND(median(glp1_claims), 0)                  AS median_glp1_claims,
            ROUND(median(glp1_share_of_diabetes), 3)       AS median_glp1_share_of_dm,
            ROUND(AVG(tirzepatide_share), 3)               AS avg_tirzepatide_share
    FROM    {H}
    WHERE   glp1_adopter
    GROUP BY ALL
    ORDER BY glp1_decile
""").df()
show(by_decile, "5d · Profile by GLP-1 decile (1 = top 10%)", precision=3)

glp1_decile,doctors,median_glp1_claims,median_glp1_share_of_dm,avg_tirzepatide_share
1,"19,983",314,0.277,0.342
2,"19,983",168,0.250,0.301
3,"19,983",116,0.236,0.275
4,"19,983",86,0.226,0.257
5,"19,983",64,0.218,0.232
6,"19,983",48,0.208,0.208
7,"19,983",35,0.198,0.187
8,"19,983",25,0.190,0.157
9,"19,983",16,0.189,0.157
10,"19,984",12,0.193,0.202


<div style="border-left:6px solid #3B82F6; background:#EAF4FF; color:#1E3A8A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">📊 Reading the output</b>

- **Tirzepatide share falls steadily with volume:** 34% in the top decile → ~16% in deciles 8–9. Heavy prescribers adopt the newer molecule first (the *early adopter* pattern from diffusion-of-innovation theory).
- The **median** GLP-1 share of diabetes Rx also falls (28% → 19%). We use the median because averages for tiny prescribers jump around: a doctor with 12 GLP-1 and 20 diabetes claims shows 60%.

</div>

<div style="border-left:6px solid #8B5CF6; background:#F3EEFF; color:#3B1F7A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

## 6 · One doctor, one row: an example profile

</div>

In [10]:
# A top-decile endocrinologist; name and exact NPI hidden (public data, shown responsibly)
example = (hcp[(hcp["specialty"] == "Endocrinology") & (hcp["glp1_decile"] == 1)]
           .sample(1, random_state=7)
           .drop(columns=["last_name", "first_name"])
           .assign(npi=lambda d: d["npi"].str[:3] + "•••••••"))
profile = example.T.reset_index()
profile[profile.columns[1]] = profile[profile.columns[1]].map(lambda v: "—" if pd.isna(v) else f"{v:,.0f}" if isinstance(v, float) and v == int(v) else f"{v:,.3f}" if isinstance(v, float) else v)
profile.columns = ["field", "value"]
show(profile.astype(str), "Example HCP 360 row (transposed)")

field,value
npi,181•••••••
city,Beverly Hills
state,MI
state_fips,26
specialty,Endocrinology
credential,MD
sex,M
practice_zip5,48025
taxonomy,207RE0101X
npi_year,2008


<div style="border-left:6px solid #3B82F6; background:#EAF4FF; color:#1E3A8A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">📊 Reading the output</b>

Everything a sales analyst or rep needs about one doctor is now in a single row: specialty, location, practice size, diabetes mix, GLP-1 brands, loyalty, deciles and (soon) industry engagement.
The **pre-visit brief** (notebook 14) and the **Ask-AI agent** (notebook 15) will read exactly this row.

</div>

<div style="border-left:6px solid #8B5CF6; background:#F3EEFF; color:#3B1F7A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

## 7 · Industry engagement (Open Payments)

</div>

In [11]:
if not pay_cols:
    print("Open Payments still downloading: re-run this notebook when "
          "data/01_raw/open_payments_industry_to_doctor/open_payments_general_*_lilly_novo.parquet exist.")
else:
    engagement = duckdb.sql(f"""
        SELECT  CASE WHEN op_lilly_payments = 0 AND op_novo_payments = 0 THEN '0 · no payments'
                     WHEN op_lilly_payments > 0 AND op_novo_payments = 0 THEN '1 · Lilly only'
                     WHEN op_lilly_payments = 0 AND op_novo_payments > 0 THEN '2 · Novo only'
                     ELSE '3 · both' END                                   AS engagement,
                COUNT(*)                                                   AS doctors,
                ROUND(median(glp1_claims), 0)                              AS median_glp1_claims,
                ROUND(AVG(tirzepatide_share), 3)                           AS avg_tirzepatide_share
        FROM    {H}
        WHERE   glp1_adopter
        GROUP BY ALL ORDER BY engagement
    """).df()
    display(show(engagement, "Engagement vs prescribing (GLP-1 prescribers)"))

engagement,doctors,median_glp1_claims,avg_tirzepatide_share
0 · no payments,"105,046",42,0.2
1 · Lilly only,"11,772",54,0.3
2 · Novo only,"20,776",59,0.2
3 · both,"62,237",87,0.3


<div style="border-left:6px solid #3B82F6; background:#EAF4FF; color:#1E3A8A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">📊 Reading the output</b>

- **53% of GLP-1 prescribers received no Lilly or Novo payment** in 2024–2025; 31% received payments from both companies.
- Engaged doctors prescribe more: median **87 GLP-1 claims** for doctors paid by both vs **42** for doctors paid by neither.
- Tirzepatide share is higher among doctors with Lilly contact (~0.3) than among those with none or Novo-only (~0.2).

</div>

<div style="border-left:6px solid #F43F5E; background:#FFF0F3; color:#881337; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">⚠️ Watch out</b>

Payments and prescribing are **correlated, not causal**. Companies engage doctors who already treat many relevant patients, so big prescribers get more contact.
We use payments as an **engagement / coverage** signal ("has anyone talked to this doctor?"), never as proof of influence.

</div>

<div style="border-left:6px solid #8B5CF6; background:#F3EEFF; color:#3B1F7A; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

## 8 · Summary

</div>

<div style="border-left:6px solid #F43F5E; background:#FFF0F3; color:#881337; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">⚠️ Limitations to carry forward</b>

1. Prescribing is **Medicare Part D 2024 only** (65+); commercial and cash-pay patients are invisible.
2. Rows under 11 claims are suppressed by CMS, so small prescribers' GLP-1 use is **under**-counted (some "non-adopters" may write a few GLP-1s).
3. Specialty comes from Part D claims; NPPES taxonomy can differ. We keep both.
4. State context is **state-level**, the same value for every doctor in a state.
5. Publication and clinical-trial links (PubMed, ClinicalTrials.gov) need fuzzy name matching; they're added in notebook 05 with match-quality checks.

</div>

<div style="border-left:6px solid #10B981; background:#EAFBF1; color:#065F46; padding:14px 18px; border-radius:10px; margin:10px 0; box-shadow:0 1px 3px rgba(0,0,0,.06);">

<b style="font-size:1.02em; letter-spacing:.3px">🎯 Key takeaways</b>

**Done:** `data/03_processed/hcp_360.parquet`: **316,559 doctors**, one row each, validated (unique NPI, totals reconcile).
**Next:** Notebook 04 (market landscape + analyst dashboard), then notebook 05 (segmentation and targeting score).

**Interview questions you can now answer**
- *What is an HCP 360 and why build one?* One row per doctor joining all sources on NPI, so targeting, dashboards and AI all read the same truth.
- *Why include doctors who don't prescribe the product?* Non-adopters with large diabetes volume are the highest-potential growth segment.
- *LEFT vs INNER JOIN?* LEFT keeps every doctor; INNER silently drops doctors missing from any source.
- *Is a missing value zero?* Depends on the source: no payment record = 0 payments; no registry record = unknown.
- *How do you catch a bad join?* Assert the key is unique and reconcile totals before and after.
- *Potential vs current value?* Diabetes volume (potential) vs GLP-1 share (current adoption): the 2×2 behind every targeting plan.

</div>